# Frozen question–answer embeddings
One vector per candidate answer, with the complete question context and all options. Uses [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B), pinned to a fixed revision. No human response tables are loaded and no weights are trained.


In [1]:
import hashlib
import json
import os
import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
OUT = ROOT / "outputs"

def read_json(path):
    return json.loads(path.read_text())

def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

config_path = ROOT / "configs/embedding_model.json"
config = read_json(config_path)
bank_path = ROOT / "data/processed/question_bank.json"
bank = read_json(bank_path)
audit_path = OUT / "data_quality_leakage_audit.json"
quality = read_json(audit_path)
assert quality["status"] == "passed_with_caveats" and all(c["passed"] for c in quality["checks"])
assert sha256(bank_path) == quality["semantic_review"]["bank_sha256"]
input_hashes = {str(p.relative_to(ROOT)): sha256(p) for p in [config_path, bank_path, audit_path]}


/Users/louismcauliffe/code/projects/AS_Tech_Task/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Build the exact texts
`question_text` already includes the original scenario/instructions, target row and full option list. Append the candidate's code and wording. IDs and split roles stay in metadata, outside the model input.


In [2]:
def candidate_text(item, option):
    return config["text_template"].format(
        instruction=config["instruction"], question_text=item["question_text"],
        code=option["code"], label=option["label"])

rows = []
for item in bank["items"]:
    for option in item["options"]:
        text = candidate_text(item, option)
        rows.append({
            "embedding_row": len(rows), "item_id": item["item_id"],
            "question_role": item["role"], "is_anchor": item["is_anchor"],
            "response_code": option["code"], "answer_index": option["index"],
            "candidate_answer": option["label"], "input_text": text,
            "text_sha256": hashlib.sha256(text.encode()).hexdigest(),
        })
index = pd.DataFrame(rows)
assert len(index) == sum(len(i["options"]) for i in bank["items"]) == 1877
assert index["item_id"].nunique() == 441
assert not index.duplicated(["item_id", "answer_index"]).any()
assert index["input_text"].is_unique
assert (index["response_code"] == index["answer_index"] + 1).all()
assert index["embedding_row"].tolist() == list(range(len(index)))
texts = index["input_text"].tolist()
print(texts[0])
display(index.groupby("question_role", sort=False).agg(questions=("item_id", "nunique"), vectors=("embedding_row", "size")))


Instruct: Represent the meaning of the candidate survey answer in the context of the question and all available options.
Query: Question:
Which part of the United States do you currently live in?

Answer options:
1. Northeast (PA, NY, NJ, RI, CT, MA, VT, NH, ME)
2. Midwest (ND, SD, NE, KS, MN, IA, MO, WI, IL, MI, IN, OH)
3. South (TX, OK, AR, LA, KY, TN, MS, AL, WV, DC, MD, DE, VA, NC, SC, GA, FL)
4. West (WA, OR, ID, MT, WY, CA, NV, UT, CO, AZ, NM)
5. Pacific (HI, AK)

Candidate answer:
1. Northeast (PA, NY, NJ, RI, CT, MA, VT, NH, ME)


,questions,vectors
question_role,,
training,430,1822
validation,8,40
test,3,15


## 2. Load and freeze Qwen
Weights stay in `.cache/`. Use the model's last-token pooling, left padding and unit-length vectors. Check token lengths before encoding so nothing is silently truncated.


In [3]:
cache = ROOT / config["cache_directory"]
snapshot = cache / ("models--" + config["model_id"].replace("/", "--")) / "snapshots" / config["revision"]
required = ["model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json",
            "modules.json", "config_sentence_transformers.json", "1_Pooling/config.json"]
if not all((snapshot / name).is_file() for name in required):
    snapshot = Path(snapshot_download(
        config["model_id"], revision=config["revision"], cache_dir=str(cache),
        allow_patterns=config["download_patterns"], token=False, max_workers=4))
assert snapshot.name == config["revision"]
model_file_hashes = {str(p.relative_to(snapshot)): sha256(p)
                     for p in sorted(snapshot.rglob("*")) if p.is_file()}

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model = SentenceTransformer(
    str(snapshot), device=device, local_files_only=True, trust_remote_code=False,
    model_kwargs={"dtype": getattr(torch, config["compute_dtype"]),
                  "attn_implementation": config["attention_implementation"]})
model.eval()
model.requires_grad_(False)
model.tokenizer.padding_side = "left"
model.max_seq_length = config["max_sequence_length"]
assert model[1].pooling_mode == "lasttoken"
assert model.get_sentence_embedding_dimension() == config["embedding_dimension"]
assert not model.training and not any(p.requires_grad for p in model.parameters())
parameter_versions = [p._version for p in model.parameters()]

encoded = model.tokenizer(texts, truncation=False, padding=False, add_special_tokens=True)
index["token_count"] = [len(ids) for ids in encoded["input_ids"]]
assert index["token_count"].max() <= model.max_seq_length <= model[0].auto_model.config.max_position_embeddings
assert (index["token_count"] > 0).all()
print(f"Device: {device}; trainable parameters: 0; longest input: {index['token_count'].max()} tokens")


/var/folders/sy/xbvsdfm92_sbg8hg3lsjl0hc0000gn/T/ipykernel_26653/1208545193.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert model.get_sentence_embedding_dimension() == config["embedding_dimension"]


Device: mps; trainable parameters: 0; longest input: 603 tokens


## 3. Encode all question–answer pairs


In [4]:
# Process similar lengths together, then put vectors back in the original index order.
order = np.argsort(-index["token_count"].to_numpy(), kind="stable")
embeddings = np.empty((len(index), config["embedding_dimension"]), dtype=config["saved_dtype"])
start = time.monotonic()
with torch.inference_mode():
    for offset in range(0, len(order), 64):
        batch_rows = order[offset:offset + 64]
        embeddings[batch_rows] = model.encode(
            [texts[int(j)] for j in batch_rows], batch_size=config["batch_size"],
            prompt="", show_progress_bar=False, convert_to_numpy=True,
            normalize_embeddings=config["normalize_embeddings"])
        completed = min(offset + 64, len(order))
        if completed % 256 == 0 or completed == len(order):
            print(f"Encoded {completed:,}/{len(order):,} candidates", flush=True)
encoding_seconds = time.monotonic() - start
print(f"Embedding matrix: {embeddings.shape}; {encoding_seconds:.1f} seconds")


Encoded 256/1,877 candidates


Encoded 512/1,877 candidates


Encoded 768/1,877 candidates


Encoded 1,024/1,877 candidates


Encoded 1,280/1,877 candidates


Encoded 1,536/1,877 candidates


Encoded 1,792/1,877 candidates


Encoded 1,877/1,877 candidates


Embedding matrix: (1877, 1024); 357.1 seconds


## 4. Verify and save
Check coverage, finite values, unit norms, distinct options and repeat encoding. These checks establish usable features, not predictive accuracy. Encoding validation/test wording with a fixed encoder uses no labels; any later PCA/scaler must fit training questions only.


In [5]:
assert embeddings.shape == (1877, 1024) and embeddings.dtype == np.float32
assert np.isfinite(embeddings).all()
norms = np.linalg.norm(embeddings, axis=1)
assert np.allclose(norms, 1.0, atol=1e-5)
assert [p._version for p in model.parameters()] == parameter_versions
assert not any(p.requires_grad or p.grad is not None for p in model.parameters())

# Every option must retain a distinct vector within its question.
separations = []
for item_id, group in index.groupby("item_id", sort=False):
    vectors = embeddings[group["embedding_row"].to_numpy()]
    assert np.unique(vectors, axis=0).shape[0] == len(group)
    distances = np.linalg.norm(vectors[:, None, :] - vectors[None, :, :], axis=-1)
    separations.extend(distances[np.triu_indices(len(group), k=1)].tolist())

# Re-encode a few training pairs of different lengths and check the saved row mapping.
probe_rows = [int(index.index[index["item_id"].eq(q)][0]) for q in ["QID11", "QID29_1", "QID84_4", "QID250_1"]]
with torch.inference_mode():
    repeated = model.encode([texts[j] for j in probe_rows], batch_size=4, prompt="",
                            show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
repeat_cosines = np.sum(repeated * embeddings[probe_rows], axis=1)
assert np.all(repeat_cosines > 0.99999)

matrix_path = OUT / "question_answer_embeddings.npy"
index_path = OUT / "question_answer_index.json"
np.save(matrix_path, embeddings, allow_pickle=False)
index_path.write_text(index.to_json(orient="records", indent=2, force_ascii=False) + "\n")
assert np.array_equal(np.load(matrix_path, allow_pickle=False), embeddings)
saved_index = read_json(index_path)
assert len(saved_index) == len(embeddings)
assert all(row["embedding_row"] == j and row["input_text"] == texts[j] for j, row in enumerate(saved_index))
assert all(sha256(ROOT / p) == h for p, h in input_hashes.items())

report = {
    "status": "passed", "model_id": config["model_id"], "revision": config["revision"],
    "model_card": config["model_card"], "model_file_sha256": model_file_hashes,
    "config": config, "input_sha256": input_hashes,
    "questions": int(index["item_id"].nunique()), "candidate_vectors": len(index),
    "embedding_shape": list(embeddings.shape), "device": str(model.device),
    "compute_dtype": str(next(model.parameters()).dtype), "saved_dtype": str(embeddings.dtype),
    "parameters": sum(p.numel() for p in model.parameters()), "trainable_parameters": 0,
    "counts_by_role": index.groupby("question_role").size().to_dict(),
    "token_counts": {"min": int(index["token_count"].min()), "max": int(index["token_count"].max()),
                     "mean": float(index["token_count"].mean())},
    "truncated_inputs": 0, "encoding_seconds": encoding_seconds,
    "norm_range": [float(norms.min()), float(norms.max())],
    "minimum_within_question_vector_distance": min(separations),
    "repeat_encoding_minimum_cosine": float(repeat_cosines.min()),
    "versions": {p: version(p) for p in ["torch", "transformers", "sentence-transformers", "huggingface-hub", "numpy"]},
    "files": {str(p.relative_to(ROOT)): {"sha256": sha256(p), "bytes": p.stat().st_size}
              for p in [matrix_path, index_path]},
    "checks": ["complete question/option coverage", "unique row-to-option mapping", "no truncation",
               "finite unit-length vectors", "distinct candidates within every question",
               "frozen parameters unchanged", "repeat encoding agreement", "saved-file round trip"],
    "human_response_tables_loaded": False, "learned_preprocessing_fitted": False,
    "next_step": "Fit dimensionality reduction on training-item vectors only, then train the shared response model."
}
report_path = OUT / "question_answer_embedding_audit.json"
report_path.write_text(json.dumps(report, indent=2) + "\n")
print(f"Saved {matrix_path.relative_to(ROOT)}, {index_path.relative_to(ROOT)} and {report_path.relative_to(ROOT)}")


Saved outputs/question_answer_embeddings.npy, outputs/question_answer_index.json and outputs/question_answer_embedding_audit.json


## 5. Retrieve a question’s answer vectors


In [6]:
def answer_vectors(item_id):
    options = index.loc[index["item_id"].eq(item_id)].sort_values("answer_index")
    if options.empty:
        raise KeyError(item_id)
    return options[["answer_index", "response_code", "candidate_answer"]], embeddings[options["embedding_row"].to_numpy()]

options, vectors = answer_vectors("QID11")
display(options)
print(f"{len(options)} options, each represented by {vectors.shape[1]} numbers.")


,answer_index,response_code,candidate_answer
0,0,1,"Northeast (PA, NY, NJ, RI, CT, MA, VT, NH, ME)"
1,1,2,"Midwest (ND, SD, NE, KS, MN, IA, MO, WI, IL, M..."
2,2,3,"South (TX, OK, AR, LA, KY, TN, MS, AL, WV, DC,..."
3,3,4,"West (WA, OR, ID, MT, WY, CA, NV, UT, CO, AZ, NM)"
4,4,5,"Pacific (HI, AK)"


5 options, each represented by 1024 numbers.
